# Notebook 01 — Environment Test
**AITR_Lab02_RushikeshShirsath_LLMFactuality**

Run this first. All assertions must pass before proceeding to experiments.

Expected time: ~2 minutes

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers==4.41.0 datasets==2.19.0 rank_bm25==0.2.2 \
    nltk==3.8.1 torch==2.3.0 accelerate==0.30.0 bitsandbytes==0.43.0 \
    scikit-learn==1.4.2 pandas==2.2.2 numpy==1.26.4 matplotlib==3.9.0 \
    seaborn==0.13.2 pyyaml==6.0.1
print('Installation complete.')

In [ ]:
# Cell 2: Check CUDA / GPU
import torch
print(f'PyTorch version:   {torch.__version__}')
print(f'CUDA available:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device:        {torch.cuda.get_device_name(0)}')
    print(f'VRAM (GB):         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 3: Check library imports
import transformers, datasets, rank_bm25, sklearn, pandas, numpy, matplotlib, seaborn
print(f'transformers: {transformers.__version__}')
print(f'datasets:     {datasets.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'pandas:       {pandas.__version__}')
print(f'numpy:        {numpy.__version__}')
print('All imports: OK')

In [ ]:
# Cell 4: Load PubMedQA and verify schema
from datasets import load_dataset

ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train')
items = list(ds)

print(f'Total items:  {len(items)}')
print(f'Keys:         {list(items[0].keys())}')
print(f'Sample label: {items[0]["final_decision"]}')
print(f'long_answer:  {items[0]["long_answer"][:100]}...')

# Verify required keys
for key in ['question', 'context', 'long_answer', 'final_decision']:
    assert key in items[0], f'Missing key: {key}'

# Verify label set
labels = {item['final_decision'] for item in items}
assert labels == {'yes', 'no', 'maybe'}, f'Unexpected labels: {labels}'

# Verify majority-class baseline
yes_count = sum(1 for item in items if item['final_decision'] == 'yes')
majority_acc = yes_count / len(items)
print(f'\nMajority-class (yes) accuracy: {majority_acc:.3f}  (expected ~0.550 — VERIFY)')
assert 0.50 <= majority_acc <= 0.60, f'Unexpected majority accuracy: {majority_acc}'

print('\nPubMedQA load: ALL CHECKS PASSED')

In [ ]:
# Cell 5: Verify test/train split (no overlap)
test_items  = items[-500:]
train_items = items[:-500]

assert len(test_items)  == 500
assert len(train_items) == len(items) - 500

# Check no overlap by pubid
test_ids  = {item['pubid'] for item in test_items}
train_ids = {item['pubid'] for item in train_items}
overlap = test_ids & train_ids
assert len(overlap) == 0, f'DATA LEAKAGE: {len(overlap)} overlapping pubids!'

print(f'Test items:  {len(test_items)}')
print(f'Train items: {len(train_items)}')
print(f'Overlap:     {len(overlap)}  (must be 0 — VERIFIED)')
print('Split verification: PASSED — no leakage')

In [ ]:
# Cell 6: Verify token F1 scorer
import sys, os
sys.path.insert(0, '/content')  # adjust if running locally

from src.evaluate import token_f1, normalize_answer, parse_mc_answer

# Token F1 unit tests
assert token_f1('the cat sat on the mat', 'the cat sat on the mat') == 1.0
assert token_f1('', 'some text') == 0.0
assert token_f1('some text', '') == 0.0
f1_partial = token_f1('aspirin reduces risk', 'aspirin reduces cardiovascular risk')
assert f1_partial > 0.5, f'Expected > 0.5, got {f1_partial}'
print(f'Partial match F1: {f1_partial:.4f}  (expected > 0.5)')

# MC parser unit tests
assert parse_mc_answer('yes, this is correct') == 'yes'
assert parse_mc_answer('No, the study found otherwise.') == 'no'
assert parse_mc_answer('maybe this could be') == 'maybe'
assert parse_mc_answer('Unclear result.') == 'maybe'  # default

print('token_f1 and parse_mc_answer: ALL UNIT TESTS PASSED')

In [ ]:
# Cell 7: BM25 index smoke test
from src.rag import build_bm25_index, retrieve

print('Building BM25 index (may take ~30s)...')
bm25, doc_texts = build_bm25_index(n_test=500)
print(f'Index built: {len(doc_texts)} documents')

# Test retrieval
test_query = 'Does aspirin reduce cardiovascular events?'
result = retrieve(bm25, doc_texts, test_query, top_k=1)
assert len(result) > 20
print(f'\nTest query: "{test_query}"')
print(f'Retrieved:  "{result[:200]}..."')
print('\nManually verify the retrieved abstract is topically relevant.')
print('BM25 smoke test: PASSED')

In [ ]:
# Cell 8: Final summary
print('=' * 55)
print('  ENVIRONMENT TEST SUMMARY')
print('=' * 55)
print(f'  CUDA GPU:          {torch.cuda.is_available()}')
print(f'  PubMedQA items:    {len(items)}')
print(f'  Majority-class:    {majority_acc:.3f}')
print(f'  Split leakage:     None (VERIFIED)')
print(f'  Token F1 scorer:   OK')
print(f'  BM25 index:        OK ({len(doc_texts)} docs)')
print('=' * 55)
print('  READY TO PROCEED TO EXPERIMENTS')
print('=' * 55)